# Weekly Project 02 - Image features

## Robot Tracking

It is recommended that you finish the exercises from Monday, before starting the project.

For this project you are given a video of some mobile robots (Robots.mp4). The task is now to track only the robots that are moving. Try to use both sparse and dense optical flow and compare the results.

For sparse optical flow, draw the tracked keypoints onto each frame and try to show the frames fast enough, such that it looks like a video.

For dense optical flow, represent the movement in any way you see fitting. For example by making a new image with the colors of each pixel representing the movement.



In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

## Sparse optical flow with Lucas-Kanade

Sparse optical flow tracks selected image features between consecutive frames. The code below keeps features whose motion is larger than `motion_threshold`, which suppresses mostly stationary background points and highlights the moving robots. Press `q` in the OpenCV window to stop playback.

In [3]:
video_path = "Robots.mp4"
cap = cv2.VideoCapture(video_path)

while True:
    ret, frame = cap.read()
    
    if not ret:
        break

    # Convert the frame to grayscale
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Display the original and grayscale frames
    cv2.imshow("Original Frame", frame)
    cv2.imshow("Grayscale Frame", gray_frame)

    # Wait for 20 ms and check if 'q' is pressed to exit
    if cv2.waitKey(20) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
video_path = "Robots.mp4"
motion_threshold = 1.5

feature_params = {
    "maxCorners": 300,
    "qualityLevel": 0.2,
    "minDistance": 7,
    "blockSize": 7,
}

lk_params = {
    "winSize": (15, 15),
    "maxLevel": 2,
    "criteria": (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
}

cap = cv2.VideoCapture(video_path)
ret, old_frame = cap.read()
if not ret:
    cap.release()
    raise FileNotFoundError(f"Could not open {video_path}")

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
trail = np.zeros_like(old_frame)
colors = np.random.randint(0, 255, (feature_params["maxCorners"], 3))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if p0 is None or len(p0) < 10:
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        trail = np.zeros_like(frame)

    if p0 is not None:
        p1, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        if p1 is not None:
            old_points = p0[status == 1]
            new_points = p1[status == 1]
            displacement = np.linalg.norm(new_points - old_points, axis=1)
            moving = displacement > motion_threshold

            for index, (new, old) in enumerate(zip(new_points[moving], old_points[moving])):
                x_new, y_new = new.astype(int)
                x_old, y_old = old.astype(int)
                color = colors[index % len(colors)].tolist()
                trail = cv2.line(trail, (x_new, y_new), (x_old, y_old), color, 2)
                frame = cv2.circle(frame, (x_new, y_new), 4, color, -1)

            p0 = new_points[moving].reshape(-1, 1, 2) if np.any(moving) else None

    output = cv2.add(frame, trail)
    cv2.imshow("Sparse optical flow - moving points", output)

    old_gray = frame_gray.copy()
    key = cv2.waitKey(20) & 0xFF
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

: 